# 03 — Old Data Feature Engineering

Leakage-aware feature engineering for the historical sensor dataset.

**Input:** `data/processed/sensor_clean_model.csv`  
**Output:** `data/processed/sensor_features.csv`

Design: preserve raw sensors, remove fully-missing sensors, add causal 30/120-minute rolling features and first differences. No imputation or full-dataset feature selection is performed here; those belong to model training.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",150)
BASE_DIR=Path("..")
PROCESSED_DATA_DIR=BASE_DIR/"data"/"processed"
INPUT_FILE=PROCESSED_DATA_DIR/"sensor_clean_model.csv"
OUTPUT_FILE=PROCESSED_DATA_DIR/"sensor_features.csv"
print(INPUT_FILE)

..\data\processed\sensor_clean_model.csv


In [2]:
df=pd.read_csv(INPUT_FILE,parse_dates=["timestamp"])
required={"timestamp","machine_status"}
missing=required-set(df.columns)
if missing: raise ValueError(f"Missing required columns: {sorted(missing)}")
df=df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
print("Shape:",df.shape)
print(df["machine_status"].value_counts(dropna=False))

Shape: (220320, 104)
machine_status
NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: count, dtype: int64


In [3]:
all_sensors=[c for c in df.columns if c.startswith("sensor_") and not c.endswith("_missing")]
missing_rate=df[all_sensors].isna().mean().sort_values(ascending=False)
dropped=missing_rate[missing_rate>=1.0].index.tolist()
sensors=[c for c in all_sensors if c not in dropped]
if not sensors: raise ValueError("No usable sensor columns found")
print("All sensors:",len(all_sensors))
print("Dropped 100% missing:",dropped)
print("Used:",len(sensors))

All sensors: 51
Dropped 100% missing: []
Used: 51


In [4]:
F=df[["timestamp","machine_status"]].copy()
V=df[sensors]
F["sensor_mean"]=V.mean(axis=1)
F["sensor_std"]=V.std(axis=1)
F["sensor_min"]=V.min(axis=1)
F["sensor_max"]=V.max(axis=1)
F["sensor_range"]=F["sensor_max"]-F["sensor_min"]
mi=[c for c in df.columns if c.endswith("_missing") and c.replace("_missing","") in sensors]
F["missing_sensor_ratio"]=df[mi].mean(axis=1) if mi else V.isna().mean(axis=1)
for s in sensors: F[s]=df[s].to_numpy()
for s in sensors: F[f"{s}_diff"]=F[s].diff()
F["sensor_mean_diff"]=F["sensor_mean"].diff()
F["sensor_std_diff"]=F["sensor_std"].diff()
F["sensor_range_diff"]=F["sensor_range"].diff()

In [5]:
for w in [30,120]:
    F[f"sensor_mean_rollmean_{w}m"]=F["sensor_mean"].rolling(w,min_periods=5).mean()
    F[f"sensor_mean_rollstd_{w}m"]=F["sensor_mean"].rolling(w,min_periods=5).std()
    F[f"sensor_std_rollmean_{w}m"]=F["sensor_std"].rolling(w,min_periods=5).mean()
    F[f"sensor_mean_rollmin_{w}m"]=F["sensor_mean"].rolling(w,min_periods=5).min()
    F[f"sensor_mean_rollmax_{w}m"]=F["sensor_mean"].rolling(w,min_periods=5).max()
features=[c for c in F.columns if c not in {"timestamp","machine_status"}]
X=F[features].select_dtypes(include=np.number)
if np.isinf(X.to_numpy()).sum(): raise ValueError("Infinite values detected")
print("Rows:",len(F),"Features:",len(features),"NaN cells:",int(X.isna().sum().sum()))

Rows: 220320 Features: 121 NaN cells: 241675


In [6]:
OUTPUT_FILE.parent.mkdir(parents=True,exist_ok=True)
F.to_csv(OUTPUT_FILE,index=False)
print("Saved:",OUTPUT_FILE)
print("Time duplicates:",F.timestamp.duplicated().sum())
print("Median interval:",F.timestamp.diff().dropna().median())
print(F.machine_status.value_counts())

Saved: ..\data\processed\sensor_features.csv
Time duplicates: 0
Median interval: 0 days 00:01:00
machine_status
NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: count, dtype: int64


### Handoff to `04_old_model_training.ipynb`
The training notebook performs chronological event splitting, training-only preprocessing, threshold tuning on validation, final test evaluation, event-level evaluation, and model/metadata export.